## Experiment design

- Get two simple prompts (clean and corrupted) made in an IOI-style but for Temporal Classification task from: *data/temporal_classification/datasets/survived_dataset_160_for_qwen3_4b_instruct_2507_templated.json*. Lets these prompts force model to answer *"long"/"short"*, to easily calculate answers' logit difference.
- Clean prompt: <code>"The goal is to become a top chef in the city. Is this a short-term or long-term goal? The answer is:"</code>
- Clean answer: <code>"long"</code>
- Corrupted prompt: <code>"The goal is to cook a warm dinner for the family. Is this a short-term or long-term goal? The answer is:"</code>
- Corrupted answer: <code>"short"</code>
- All 160 examples of such prompts are aligned by number of tokens for Qwen3-4B-Instruct-2507.
- Run activation patching from TransformerLens.

## Experiment results

In [ ]:
from pathlib import Path
import sys
import json
import os

IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except:
  IN_COLAB = False
  print("Not running as a Colab notebook")

if IN_COLAB:
  %pip install transformer_lens
  %pip install memory_profiler
  %load_ext memory_profiler
  from google.colab import drive
  drive.mount('/content/drive')
  ! pwd # Returns /content
  ! cp "/content/drive/MyDrive/Colab Notebooks/patching_algorithms.py" /content/patching_algorithms.py
  ! cp "/content/drive/MyDrive/Colab Notebooks/survived_dataset_160_for_qwen3_4b_instruct_2507_templated.json" /content/temporal_scope_for_activation_patching.json
  import patching_algorithms

import plotly.express as px
import plotly.io as pio
import pandas as pd

Running as a Colab notebook
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.3/945.3 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 9.2 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=b2bd21c69bd554f2bc6fb12ce256d65b379081a3410ca1c7873fc4c361482b4e
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator
Mounted at /content/drive
/content


### 1. Define the model of interest

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

### 2. Read the dataset, prepare inputs for "long" circuits

In [ ]:
temporal_dataset = None
dataset_path = "temporal_scope_for_activation_patching.json"
with open(dataset_path) as f:
    temporal_dataset = json.load(f)

clean_prompts = []
clean_answers = []
corrupted_prompts = []
corrupted_answers = []

pairs_ids = []

for sample in temporal_dataset:
    clean_question = sample["corrupted"]["question"]
    clean_prompts.append(clean_question)
    clean_answers.append(sample["corrupted"]["answer"])

    corrupted_question = sample["clean"]["question"]
    corrupted_prompts.append(corrupted_question)
    corrupted_answers.append(sample["clean"]["answer"])

    pairs_ids.append(sample["id"])


In [ ]:
from transformer_lens import (
    loading_from_pretrained,
    HookedTransformer,
    ActivationCache,
    patching
)

loading_from_pretrained.OFFICIAL_MODEL_NAMES.append(MODEL_NAME)

model_name = MODEL_NAME

### Run Activation patching function

In [ ]:
import torch

class ExperimentResult:
  def __init__(self, name, metric_name, technique_name, layers_pos_scores):
    self.name = name
    self.metric_name = metric_name
    self.technique_name = technique_name
    self.layers_pos_scores = layers_pos_scores
    self.layers_scores_avg = torch.mean(self.layers_pos_scores, dim=1)

def run_full_activation_patching_experiment(model_name, clean_prompts, clean_answers, corrupted_prompts, corrupted_answers):
  techniques_to_run = [("Denoising_Optimal", patching_algorithms.ActivationPatching.Technique.DENOISING_OPTIMAL)]
                      # ("Noising_Optimal", patching_algorithms.ActivationPatching.Technique.NOISING_OPTIMAL)]
  residual_results = []
  attn_out_results = []
  mlp_out_results = []
  for technique_name, technique_type in techniques_to_run:
      print(f"\n\n\nRunning experiment for technique: {technique_name}!")
      act_patch = patching_algorithms.ActivationPatching(MODEL_NAME,
                                                        clean_prompts, clean_answers,
                                                        corrupted_prompts, corrupted_answers, patching_algorithms.ActivationPatching.Metric.LOGIT_DIFF,
                                                        technique_type, patching_algorithms.ActivationPatching.Viz.READER_FRIENDLY, unbatched=True, pairs_ids=pairs_ids, dump=True)
      print("\n\n\nRunning residual patching...")
      result = act_patch.patch_residual()
      residual_results.append(ExperimentResult(f"residual_logit_diff_{technique_name}", "logit_diff", technique_name, result[0]))
      residual_results.append(ExperimentResult(f"residual_logprob_clean_{technique_name}", "logprob_clean", technique_name, result[1]))
      residual_results.append(ExperimentResult(f"residual_logprob_corrupted_{technique_name}", "logprob_corrupted", technique_name, result[2]))
      residual_results.append(ExperimentResult(f"residual_logit_clean_{technique_name}", "logit_clean", technique_name, result[3]))
      residual_results.append(ExperimentResult(f"residual_logit_corrupted_{technique_name}", "logit_corrupted", technique_name, result[4]))
      print("Residual patching - Done.")
      print("\n\n\nRunning attention output patching...")
      attn_result = act_patch.patch_attn_out()
      attn_out_results.append(ExperimentResult(f"attn_out_logit_diff_{technique_name}", "logit_diff", technique_name, attn_result[0]))
      attn_out_results.append(ExperimentResult(f"attn_out_logprob_clean_{technique_name}", "logprob_clean", technique_name, attn_result[1]))
      attn_out_results.append(ExperimentResult(f"attn_out_logprob_corrupted_{technique_name}", "logprob_corrupted", technique_name, attn_result[2]))
      attn_out_results.append(ExperimentResult(f"attn_out_logit_clean_{technique_name}", "logit_clean", technique_name, attn_result[3]))
      attn_out_results.append(ExperimentResult(f"attn_out_logit_corrupted_{technique_name}", "logit_corrupted", technique_name, attn_result[4]))
      print("Attention output patching - Done.")
      print("\n\n\nRunning MLP output patching...")
      mlp_result = act_patch.patch_mlp_out()
      mlp_out_results.append(ExperimentResult(f"mlp_out_logit_diff_{technique_name}", "logit_diff", technique_name, mlp_result[0]))
      mlp_out_results.append(ExperimentResult(f"mlp_out_logprob_clean_{technique_name}", "logprob_clean", technique_name, mlp_result[1]))
      mlp_out_results.append(ExperimentResult(f"mlp_out_logprob_corrupted_{technique_name}", "logprob_corrupted", technique_name, mlp_result[2]))
      mlp_out_results.append(ExperimentResult(f"mlp_out_logit_clean_{technique_name}", "logit_clean", technique_name, mlp_result[3]))
      mlp_out_results.append(ExperimentResult(f"mlp_out_logit_corrupted_{technique_name}", "logit_corrupted", technique_name, mlp_result[4]))
      print("MLP output patching - Done.")

      print("Cleaning the memory...")
      del act_patch

      import gc
      gc.collect()
  return residual_results, attn_out_results, mlp_out_results

Run full Activation patching experiment for "long" clean prompts and "short" corrupted ones.
- The experiment is run for three metrics: *logit_diff*, *logprob* and *logit* (Correct for mean = 0 in logit experiment!).
- The experiment is run for denoising and noising techniques.

In [ ]:
clean_long_res, clean_long_attn, clean_long_mlp = run_full_activation_patching_experiment(MODEL_NAME, clean_prompts, clean_answers, corrupted_prompts, corrupted_answers)